# 1. SKILL文件与按需加载

第十一章用 MCP 解决“能力从哪里来”：服务提供工具，Agent 调用它们。但有些任务需要的是**一套做事的流程和一份参考资料**。例如套餐咨询：先问清团队人数和用量，再查选型指南，按规则推荐最便宜的合适套餐。

把这些全写进系统提示，每次对话都会带上它们，即使用户只是想翻译一句话。**Skill** 把流程和资料放进一个目录，平时只让模型知道“有这个技能”，需要时再加载。

学习目标：

1. 说出 Skill 与系统提示、Tool、MCP 的分工。
2. 按 Agent Skills 规范编写 `SKILL.md`，并引用一份辅助资料。
3. 用 `create_agent` 加两个自定义工具实现最小加载器。
4. 用三个任务观察每一层内容在什么时候进入上下文。

本节会请求聊天模型：1.5 的三个任务各运行一次 Agent，沿用第十一章的 `ChatDeepSeek` 配置；其余实验不请求模型。选择项目 `.venv` 内核后，按顺序执行。

## 1.1 Skill解决什么问题

本章的 Skill 指 **Agent Skills**：一个开放的目录格式规范，以 `SKILL.md` 为核心，把某类任务的流程说明、领域资料和辅助文件组织在一起。

| 方式 | 提供什么 | 何时进入模型上下文 | 本章例子 |
|---|---|---|---|
| 系统提示 | 固定的身份和规则 | 每次请求都在 | “你是 atguigu助手的客服” |
| Tool | 一个可以执行的操作 | 名称、描述、参数 Schema 每次都在；执行结果在调用之后 | 第十一章的 `multiply` |
| MCP | 由服务提供工具等能力的协议 | 工具部分与 Tool 相同 | 第十一章的套餐服务 |
| Skill | 一类任务的流程说明与辅助资料 | 名称和描述每次都在；正文和资料按需加载 | 本节的套餐咨询技能 |

Skill 本身不执行任何操作。它告诉模型**怎样完成一类任务**：先做什么、何时读哪份资料、该调用哪些工具。真正执行操作的仍然是 Tool 或 MCP 工具。

![一个 Skill 分三层进入上下文](assets/01-skill-three-layers.svg)

规范把这种分层加载称为**渐进式披露**：元数据常驻，正文在任务相关时加载，资料在确实需要时读取。本节的实验就是验证这三层。

## 1.2 环境准备

读取 `SKILL.md` 开头的 YAML 元数据用 `yaml.safe_load`。PyYAML 是 langchain-core 的依赖，项目环境中已经安装。模型配置与第十一章相同，初始化模型对象还不会发送请求。

In [8]:
import json
import os
import re
import shutil
import sys
from importlib.metadata import version
from pathlib import Path
from tempfile import TemporaryDirectory

import yaml
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 lang-chain-learn 项目或其子目录运行本笔记。")

SKILLS_DIR = PROJECT_ROOT / "Chapter12_skill" / "skills"
PLAN_SKILL_DIR = SKILLS_DIR / "plan-consultation"
DESIGN_SKILL_DIR = SKILLS_DIR / "frontend-design"
ENV_FILE = PROJECT_ROOT / "conf" / ".env"
load_dotenv(dotenv_path=ENV_FILE, override=True)

print("Python:", sys.executable)
for package in ["langchain", "langchain-deepseek", "pyyaml"]:
    print(f"{package}: {version(package)}")
print("技能目录存在:", SKILLS_DIR.is_dir())

Python: /Users/skk/Developer/lang-chain-learn/.venv/bin/python
langchain: 1.4.3
langchain-deepseek: 1.1.1
pyyaml: 6.0.3
技能目录存在: True


In [9]:
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(f"请在 {ENV_FILE} 中配置 OPENAI_API_KEY=，再运行 Agent 部分。")

MODEL_NAME = "gpt-6-luna"
model = ChatOpenAI(
    model=MODEL_NAME,
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=os.environ["OPENAI_API_BASE"],
    #profile=custom_profile
)
print("模型:", type(model).__name__, MODEL_NAME)

模型: ChatOpenAI gpt-6-luna


## 1.3 编写套餐咨询Skill

### 1.3.1 技能目录

项目中已保存 `skills/plan-consultation/`。一个技能就是一个目录：`SKILL.md` 必须有，其他文件按需添加。规范建议的子目录有 `references/`（参考资料）、`assets/`（模板等静态资源）和 `scripts/`（可执行脚本），本节只用 `references/`。

In [10]:
for path in sorted(SKILLS_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(SKILLS_DIR))

frontend-design/LICENSE.txt
frontend-design/SKILL.md
plan-consultation/SKILL.md
plan-consultation/references/plan-guide.md


### 1.3.2 SKILL.md：元数据加正文

`SKILL.md` 分两部分：两行 `---` 之间是 YAML 元数据，之后是 Markdown 正文。

In [12]:
print((PLAN_SKILL_DIR / "SKILL.md").read_text(encoding="utf-8"))
print("============="*10)
print((DESIGN_SKILL_DIR / "SKILL.md").read_text(encoding="utf-8"))

---
name: plan-consultation
description: 为咨询 atguigu助手标准订阅套餐的用户推荐套餐并估算月费。用户询问该选哪个套餐，或问到套餐价格、成员人数上限、AI 问答额度、API 调用额度时使用。
metadata:
  author: lang-chain-learn
  version: "1.0"
---

# 套餐咨询

帮助用户在试用版、基础版、专业版、企业版中，选出满足需求且价格最低的套餐。

## 执行步骤

1. 确认三项信息：团队成员人数、每月 AI 问答次数、是否需要 API 调用。
   缺少任何一项时，先向用户追问缺少的信息，不要读取资料，也不要推荐套餐。
2. 三项信息齐全后，读取 [套餐选型指南](references/plan-guide.md)，查出各套餐的成员人数上限、月度 AI 问答额度和 API 调用额度。
3. 从试用版开始，按价格从低到高逐个比较，选出第一个三项需求都满足的套餐。专业版也不满足时，推荐企业版，并说明需要联系销售。
4. 按“单价 × 成员人数”估算月费。企业版不公开标价，不做估算。

## 输出要求

- 推荐套餐：套餐名称
- 选择理由：逐项对比三项需求与套餐的上限或额度
- 月费估算：写出计算式
- 需要人工确认：选型指南没有写明的内容

只依据选型指南中的数字回答，不要凭印象补充价格或额度。

---
name: frontend-design
description: Guidance for distinctive, intentional visual design when building new UI or reshaping an existing one. Helps with aesthetic direction, typography, and making choices that don't read as templated defaults.
license: Complete terms in LICENSE.txt
---

# Frontend Design

Approach this as the design lead at a design studio known for giving every client a 

**元数据**中的字段（规范规定）：

| 字段 | 是否必填 | 规则 |
|---|---|---|
| `name` | 必填 | 1–64 个字符，只能用小写字母、数字和连字符，不能以连字符开头或结尾，不能连续两个连字符；**必须与目录名相同** |
| `description` | 必填 | 1–1024 个字符，写清“做什么”和“什么时候用”，并包含用户可能提到的关键词 |
| `metadata` | 可选 | 字符串到字符串的映射，存放作者、版本等附加信息 |
| `license`、`compatibility`、`allowed-tools` | 可选 | 许可证、运行环境要求、预先批准的工具；本节不用 |

`description` 是模型决定是否加载技能的**唯一依据**，所以它列出了“选哪个套餐”“价格”“人数上限”等用户会说的话。

**正文**没有固定格式，本例写了执行步骤和输出要求。第 1 步明确规定：信息不全时先追问，不读资料。第 2 步用相对路径 `references/plan-guide.md` 指向资料，规范要求引用路径相对于技能目录，且只引用一层。

### 1.3.3 辅助资料：只在需要具体数字时读取

选型指南摘自第十章知识库的“二、套餐说明”。具体数字放在单独的文件里，正文保持简短，只有真正需要比较数字时才读取。

In [13]:
print((PLAN_SKILL_DIR / "references" / "plan-guide.md").read_text(encoding="utf-8"))

# 套餐选型指南

摘自第十章 `knowledge.txt` 的“二、套餐说明”（2026 Q1 版），仅用于教学。

| 套餐 | 价格 | 成员人数上限 | 月度 AI 问答额度 | API 调用额度 | OCR 图片解析 |
|---|---|---|---|---|---|
| 试用版 | 0 元，注册后 14 天 | 1 人 | 200 次 | 不支持 | 不支持 |
| 基础版 | 99 元 / 用户 / 月 | 10 人 | 5000 次 | 每月 10000 次 | 每月 200 页 |
| 专业版 | 199 元 / 用户 / 月 | 50 人 | 30000 次 | 每月 80000 次 | 每月 2000 页 |
| 企业版 | 按年签约，不公开标价 | 按合同约定 | 按合同约定 | 按合同约定 | 按合同约定 |

补充说明：

- 价格按每位用户每月计算。
- AI 问答额度按自然月统计，每月 1 日重置，未用完的额度不结转。额度用尽后停止问答服务，直到下月重置或升级套餐。
- 企业版另有 SSO 单点登录、私有模型路由、专属客户成功经理和 SLA 服务承诺，具体内容以合同为准。



## 1.4 实现最小加载器

只创建 `SKILL.md` 还不够：`create_agent` 不会扫描技能目录，当前 langchain 1.4.3 也没有内置的 Skill 接口。LangChain 文档的 Skills 模式是**自己写一个 `load_skill` 工具**，本节照此实现三层：

| 层次 | 负责的代码 | 进入上下文的方式 |
|---|---|---|
| 第 1 层 元数据 | `discover_skills` + `build_skills_prompt` | 写进系统提示 |
| 第 2 层 正文 | 工具 `load_skill` | 模型调用后成为 ToolMessage |
| 第 3 层 资料 | 工具 `read_skill_file` | 模型调用后成为 ToolMessage |

### 1.4.1 发现技能：只取元数据

`split_skill_file` 把 `SKILL.md` 拆成元数据字典和正文。`discover_skills` 扫描 `skills/*/SKILL.md`，按规范检查 `name` 和 `description`，只保留名称、描述和目录，正文这时不使用。

In [14]:
NAME_PATTERN = re.compile(r"^[a-z0-9]+(-[a-z0-9]+)*$")


def split_skill_file(skill_file: Path) -> tuple[dict, str]:
    """把 SKILL.md 拆成 YAML 元数据字典和 Markdown 正文。"""
    text = skill_file.read_text(encoding="utf-8")
    if not text.startswith("---\n"):
        raise ValueError(f"{skill_file} 缺少开头的 YAML 元数据")
    # 需要限制次数,防止后续再度出现
    _, frontmatter, body = text.split("---\n", 2)
    # 原版的 yaml.load() 可能会解析并执行 YAML 中嵌入的任意 Python 代码或对象（例如通过 !!python/object/apply 标签），从而带来严重的安全隐患（类似反序列化漏洞）。而 safe_load() 会拒绝这些不安全的自定义标签。
    return yaml.safe_load(frontmatter), body.strip()


def discover_skills(skills_dir: Path) -> dict[str, dict]:
    """扫描技能目录，检查元数据，只保留名称、描述和目录。"""
    skills = {}
    for skill_file in sorted(skills_dir.glob("*/SKILL.md")):
        metadata, _ = split_skill_file(skill_file)
        name = metadata.get("name", "")
        description = metadata.get("description", "")
        problems = []
        # 规则与 OpenAI 的官方文档一致
        if not (len(name) <= 64 and NAME_PATTERN.fullmatch(name)):
            problems.append("name 只能用小写字母、数字和单个连字符，长度 1–64")
        if name != skill_file.parent.name:
            problems.append("name 必须与目录名相同")
        if not 1 <= len(description) <= 1024:
            problems.append("description 长度应为 1–1024")
        if problems:
            raise ValueError(f"{skill_file.parent.name}/SKILL.md：" + "；".join(problems))
        skills[name] = {"description": description, "dir": skill_file.parent}
    return skills


SKILLS = discover_skills(SKILLS_DIR)
for name, info in SKILLS.items():
    print("名称:", name)
    print("描述:", info["description"])
    print("目录:", info["dir"].relative_to(PROJECT_ROOT))

名称: frontend-design
描述: Guidance for distinctive, intentional visual design when building new UI or reshaping an existing one. Helps with aesthetic direction, typography, and making choices that don't read as templated defaults.
目录: Chapter12_skill/skills/frontend-design
名称: plan-consultation
描述: 为咨询 atguigu助手标准订阅套餐的用户推荐套餐并估算月费。用户询问该选哪个套餐，或问到套餐价格、成员人数上限、AI 问答额度、API 调用额度时使用。
目录: Chapter12_skill/skills/plan-consultation


**修改一个条件**：把技能复制到临时目录，只把 `name` 改成 `Plan_Consultation`，其余不变。先预测检查结果。

In [15]:
with TemporaryDirectory(prefix="skills-") as temporary_directory:
    copied_dir = Path(temporary_directory) / "plan-consultation"
    shutil.copytree(PLAN_SKILL_DIR, copied_dir)
    skill_file = copied_dir / "SKILL.md"
    skill_file.write_text(
        skill_file.read_text(encoding="utf-8").replace(
            "name: plan-consultation", "name: Plan_Consultation"
        ),
        encoding="utf-8",
    )
    try:
        discover_skills(Path(temporary_directory))
    except ValueError as exc:
        print("检查未通过:", exc)

检查未通过: plan-consultation/SKILL.md：name 只能用小写字母、数字和单个连字符，长度 1–64；name 必须与目录名相同


大写字母和下划线不符合命名规则，名称也与目录名不一致，两条问题都被报告。检查放在发现阶段，有问题的技能不会进入系统提示。

### 1.4.2 第1层：把技能摘要放进系统提示

`build_skills_prompt` 把每个技能写成一行“名称：描述”，并说明使用规则：相关时先调用 `load_skill`，无关时直接回答。

In [16]:
BASE_PROMPT = "你是 atguigu助手的客服，用中文简短回答。"


def build_skills_prompt(skills: dict[str, dict]) -> str:
    lines = [
        "## 可用技能",
        "处理用户请求前，先判断它是否与下列技能相关：",
        "- 相关时，先调用 load_skill 加载该技能的完整说明，再按说明执行；",
        "- 无关时直接回答，不要加载技能。",
        "",
    ]
    lines += [f"- {name}：{info['description']}" for name, info in skills.items()]
    return "\n".join(lines)


SYSTEM_PROMPT = BASE_PROMPT + "\n\n" + build_skills_prompt(SKILLS)
print(SYSTEM_PROMPT)
print()
print("系统提示字符数:", len(SYSTEM_PROMPT))

你是 atguigu助手的客服，用中文简短回答。

## 可用技能
处理用户请求前，先判断它是否与下列技能相关：
- 相关时，先调用 load_skill 加载该技能的完整说明，再按说明执行；
- 无关时直接回答，不要加载技能。

- frontend-design：Guidance for distinctive, intentional visual design when building new UI or reshaping an existing one. Helps with aesthetic direction, typography, and making choices that don't read as templated defaults.
- plan-consultation：为咨询 atguigu助手标准订阅套餐的用户推荐套餐并估算月费。用户询问该选哪个套餐，或问到套餐价格、成员人数上限、AI 问答额度、API 调用额度时使用。

系统提示字符数: 437


### 1.4.3 第2、3层：两个加载工具

| 工具 | 参数 | 返回值 |
|---|---|---|
| `load_skill` | `skill_name`：技能名称 | `SKILL.md` 的正文，前面加一句说明；名称不存在时返回可用技能列表 |
| `read_skill_file` | `skill_name`，`path`：正文中写的相对路径 | 文件内容；路径不在该技能目录内或文件不存在时返回说明 |

`path` 由模型填写，可能写成 `../` 开头的路径。`read_skill_file` 先用 `resolve()` 求出真实路径，再用 `is_relative_to` 确认它仍在技能目录内，避免模型读到技能目录之外的文件。

两个工具都在找不到时**返回说明文字**而不是抛出异常，模型可以据此改正名称或路径。

In [ ]:
@tool
def load_skill(skill_name: str) -> str:
    """加载一个技能的完整说明（SKILL.md 正文）。skill_name 使用“可用技能”中的名称。"""
    info = SKILLS.get(skill_name)
    if info is None:
        return f"没有名为 {skill_name} 的技能。可用技能：{', '.join(SKILLS)}"
    _, body = split_skill_file(info["dir"] / "SKILL.md")
    return f"已加载技能 {skill_name}。正文中的相对路径可以用 read_skill_file 读取。\n\n{body}"


@tool
def read_skill_file(skill_name: str, path: str) -> str:
    """
    读取技能目录中的辅助文件。path 使用技能说明中写的相对路径，例如 references/plan-guide.md。
    
    Args:
        skill_name: 技能名称,便于取skill所在的目录
        path: 相对路径

    """
    info = SKILLS.get(skill_name)
    if info is None:
        return f"没有名为 {skill_name} 的技能。"
    skill_dir = info["dir"].resolve()
    print("skill_dir:", skill_dir)
    target = (skill_dir / path).resolve()
    if not target.is_relative_to(skill_dir) or not target.is_file():
        return f"技能 {skill_name} 的目录中没有文件 {path}。"
    return target.read_text(encoding="utf-8")

先不经过模型，直接调用两个工具，确认返回内容，再试两个错误输入。`../../README.md` 指向第十二章的 README，文件确实存在，但不在技能目录内。

In [24]:
loaded_body = load_skill.invoke({"skill_name": "plan-consultation"})
print("load_skill 返回字符数:", len(loaded_body))
print(loaded_body[:90], "……")
print()

guide_text = read_skill_file.invoke(
    {"skill_name": "plan-consultation", "path": "references/plan-guide.md"}
)
print("read_skill_file 返回字符数:", len(guide_text))
print(guide_text.splitlines()[0])
print()

outside_path = "../../README.md"
print("文件实际存在:", (PLAN_SKILL_DIR / outside_path).resolve().is_file())
print("越出技能目录:", read_skill_file.invoke(
    {"skill_name": "plan-consultation", "path": outside_path}
))
print("技能名不存在:", load_skill.invoke({"skill_name": "refund-review"}))

load_skill 返回字符数: 485
已加载技能 plan-consultation。正文中的相对路径可以用 read_skill_file 读取。

# 套餐咨询

帮助用户在试用版、基础版、专业版、企业版中，选出满 ……

skill_dir: /Users/skk/Developer/lang-chain-learn/Chapter12_skill/skills/plan-consultation
read_skill_file 返回字符数: 513
# 套餐选型指南

文件实际存在: True
skill_dir: /Users/skk/Developer/lang-chain-learn/Chapter12_skill/skills/plan-consultation
越出技能目录: 技能 plan-consultation 的目录中没有文件 ../../README.md。
技能名不存在: 没有名为 refund-review 的技能。可用技能：frontend-design, plan-consultation


### 1.4.4 创建Agent

加载器就是“系统提示 + 两个工具”，写法与第七章相同。

In [25]:
skill_agent = create_agent(
    model=model,
    tools=[load_skill, read_skill_file],
    system_prompt=SYSTEM_PROMPT,
)
print("Agent 的工具:", [load_skill.name, read_skill_file.name])

Agent 的工具: ['load_skill', 'read_skill_file']


## 1.5 三个任务分别加载了什么

![三个任务，加载到不同的层](assets/02-three-tasks-loading.svg)

三个任务从无关到需要资料。先按图预测每个任务会走到哪一层，再运行验证。

`run_task` 运行一次 Agent。`show_loading` 打印加载工具的调用、每个工具结果的字符数，以及每次模型请求的输入 token（来自 `AIMessage.usage_metadata`），并返回统计结果，供 1.6 对比。

In [33]:
from rich import print as rprint
async def run_task(question: str) -> list:
    # 处理疑问
    result = await skill_agent.ainvoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 10},
    )
    rprint(f"本次{question}的回答是")
    rprint(result)
    return result["messages"]


def show_loading(messages: list) -> dict:
    """打印加载工具的调用与模型输入 token，返回调用次数和加载的字符数。"""
    loaded_chars = 0
    for message in messages:
        if isinstance(message, AIMessage):
            # 获取工具使用的 token 或者回答的token
            input_tokens = (message.usage_metadata or {}).get("input_tokens")
            # 遍历工具调用
            for call in message.tool_calls:
                print(f"[AIMessage] 输入 {input_tokens} token，请求 {call['name']}",
                      json.dumps(call["args"], ensure_ascii=False))
            if not message.tool_calls:
                print(f"[AIMessage] 输入 {input_tokens} token，回答:")
                print(message.text)
        elif isinstance(message, ToolMessage):
            loaded_chars += len(message.text)
            print(f"[ToolMessage] {message.name} 返回 {len(message.text)} 字符")
    called = [call["name"] for message in messages
              if isinstance(message, AIMessage) for call in message.tool_calls]
    return {
        "load_skill": called.count("load_skill"),
        "read_skill_file": called.count("read_skill_file"),
        "loaded_chars": loaded_chars,
    }

### 1.5.1 任务1：无关任务

In [34]:
unrelated_stats = show_loading(await run_task("把“谢谢你的帮助”翻译成英文。"))

本次把“谢谢你的帮助”翻译成英文。的回答是

{
    'messages': [
        HumanMessage(
            content='把“谢谢你的帮助”翻译成英文。',
            additional_kwargs={},
            response_metadata={},
            id='1eea9e5f-9077-402e-8d7d-6ce107c6833f'
        ),
        AIMessage(
            content=[
                {
                    'type': 'text',
                    'text': 'Thank you for your help.',
                    'annotations': [],
                    'id': 'msg_0b94fd17e55f1a4a016accef98930087d0a2d69d9df1112af1',
                    'phase': 'final_answer'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0b94fd17e55f1a4a016accef96ca5487d0aa32bec68725a750',
                'created_at': 1791815574.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0b94fd17e55f1a4a016accef96ca5487d0aa32bec68725a750',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4661,
                'output_tokens': 10,
                'total_tokens': 4671,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        )
    ]
}

[AIMessage] 输入 4661 token，回答:
Thank you for your help.


模型只看到一行技能摘要，判断翻译与套餐无关，没有调用加载工具。技能正文和资料都没有进入上下文。

### 1.5.2 任务2：相关任务，但信息不全

In [35]:
partial_stats = show_loading(await run_task("我们想开通你们的套餐，选哪个比较合适？"))

本次我们想开通你们的套餐，选哪个比较合适？的回答是

{
    'messages': [
        HumanMessage(
            content='我们想开通你们的套餐，选哪个比较合适？',
            additional_kwargs={},
            response_metadata={},
            id='e202abe8-f46b-46e4-920b-56758f146a42'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"skill_name":"plan-consultation"}',
                    'call_id': 'call_7d6e4fccc1174acdbbbe461968e314bd',
                    'name': 'load_skill',
                    'type': 'function_call',
                    'id': 'fc_08a5e39f42ed2410016accefaa66648191ad2aa55c6d981c97',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_08a5e39f42ed2410016accefa99664819182a577e55464344d',
                'created_at': 1791815593.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_08a5e39f42ed2410016accefa99664819182a577e55464344d',
            tool_calls=[
                {
                    'name': 'load_skill',
                    'args': {'skill_name': 'plan-consultation'},
                    'id': 'call_7d6e4fccc1174acdbbbe461968e314bd',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4667,
                'output_tokens': 22,
                'total_tokens': 4689,
                'input_token_details': {'cache_creation': 0, 'cache_read': 3584},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='已加载技能 plan-consultation。正文中的相对路径可以用 read_skill_file 读取。\n\n# 
套餐咨询\n\n帮助用户在试用版、基础版、专业版、企业版中，选出满足需求且价格最低的套餐。\n\n## 执行步骤\n\n1. 
确认三项信息：团队成员人数、每月 AI 问答次数、是否需要 API 调用。\n   
缺少任何一项时，先向用户追问缺少的信息，不要读取资料，也不要推荐套餐。\n2. 三项信息齐全后，读取 
[套餐选型指南](references/plan-guide.md)，查出各套餐的成员人数上限、月度 AI 问答额度和 API 调用额度。\n3. 
从试用版开始，按价格从低到高逐个比较，选出第一个三项需求都满足的套餐。专业版也不满足时，推荐企业版，并说明需要联系
销售。\n4. 按“单价 × 成员人数”估算月费。企业版不公开标价，不做估算。\n\n## 输出要求\n\n- 推荐套餐：套餐名称\n- 
选择理由：逐项对比三项需求与套餐的上限或额度\n- 月费估算：写出计算式\n- 
需要人工确认：选型指南没有写明的内容\n\n只依据选型指南中的数字回答，不要凭印象补充价格或额度。',
            name='load_skill',
            id='56892dd1-c8cd-490f-b470-df1c1fa8c440',
            tool_call_id='call_7d6e4fccc1174acdbbbe461968e314bd'
        ),
        AIMessage(
            content=[
                {
                    'id': 'rs_0a8ede39e337c7cd006accefacef3c87d1a2954dc3329b94e0',
                    'summary': [],
                    'type': 'reasoning',
                    'encrypted_content': 
'gAAAAABqzO-tXYgtsLltKwyacdS1REIagqZtEFWfBc9JkGZ8VHHPKi0RGMnD9tTLhQ48UldvDzWJZBOzJnyF4qsDdem4DCp0rXe4kjHqLIPFZpL9oH
Vtgn5GwsrBs931BRC2yle0Eeifn_24pr78sKynr2unbEvfW98oR4vF-HKj6ks-GMWDcUnTyE4grrtNUe8wwXyU1JBSo5jpsgE0r8cSHM9pA8hqZIvL8
q-KE8rvgd-DSbzXYHGwXAjJHj5Vk9FEUotMa6MKPxYL4Dcr0jlUwYmD-KQIFVBOvaPjofVKFoqZ5Z0a96Ko4Z1Mn2v_TSeRWmWmZG_i_NMCbz77Q-BU
EFicCjqDpWu23BgD9_TjsgxeHWMKtqDrELbY6QAXIXyCATquO0-g5A-seoqwlWZiwQ9QOMjuzdQ4svdmTiBuZwGT9sKqF4T0UQZiMoCNjeqzIR_nw-w
l-Qv4WVpc69Qij16TQLYWCpl7BW2yX596b6rVw8C2D0FYJs2EDumeFubC7GDUaNCeCnczeCEwKpjO9ngI9M-QPn_w6WW8Up_o-ORuufD9UxEjelrMku
ty1dXuDTAJp8ZC88ggR5iMeSyZYimvFfeUcbUux2jn8tcu9YCMnaCuaLRx0M8WHklpKBKMM_KlRp9gXtWlqRvpew4hV9rFtrsFZo0ttaBUp5Y1zDzIm
e5k37uIcJDwEtUOrnoxAAr2_mQwDOtXWJWEaHHihC7GunwjQ3MmZVRDBi1Z5LsxGihHJZIOSuslDdQ2av1MAxHAA_ujnkGjkI7PsQTwg5ezvf8ODdgj
IUYbgEZER9dMOns2jFyTt9KeMxDZTEaFfaqQpC3rj9Cb62GRi-xayU8O9Iv3yB0kKiGSczz12JxlDvk2lWdU-0AmqvAKfWwsNPOqU8VGdvM4_SoC0p2
_PQ1tjQmjlLU-nlJr095QtC1vvvislDeRZ_PqqTURGrbv-eEhc1d0x2dAXim70kuWKNzWMzjx5GwQiSt5VPW27X_SU5Gm_hmyV4KESC6Y5gxio1qIx6
Zy9neARSc1eH2zBWj5mZq71BrcwvaAfjb0hpiEYc6pxBp7aqoAk1YJJHiuc9hVQV26xgoKsSdWjmyvnawCxQGo5nFjobWim8bqAt0gLWxQODPJFGFAm
81Akm

[AIMessage] 输入 4667 token，请求 load_skill {"skill_name": "plan-consultation"}
[ToolMessage] load_skill 返回 485 字符
[AIMessage] 输入 631 token，回答:
为了帮你选到满足需求且价格最低的套餐，请告诉我：

1. 团队有多少位成员？
2. 每月预计需要多少次 AI 问答？
3. 是否需要 API 调用？


模型调用 `load_skill` 读取正文，再按第 1 步追问缺少的信息。它没有读取选型指南：正文规定信息不全时不读资料。加载正文后，下一次模型请求的输入 token 明显增加，增加的主要就是正文；没有加载的资料不占用上下文。

### 1.5.3 任务3：信息齐全，需要辅助资料

In [36]:
complete_stats = show_loading(await run_task(
    "我们团队 12 个人，每月 AI 问答大约 2 万次，还需要调用 API。选哪个套餐？每月大约多少钱？"
))

skill_dir: /Users/skk/Developer/lang-chain-learn/Chapter12_skill/skills/plan-consultation


本次我们团队 12 个人，每月 AI 问答大约 2 万次，还需要调用 API。选哪个套餐？每月大约多少钱？的回答是

{
    'messages': [
        HumanMessage(
            content='我们团队 12 个人，每月 AI 问答大约 2 万次，还需要调用 API。选哪个套餐？每月大约多少钱？',
            additional_kwargs={},
            response_metadata={},
            id='9fff4ef5-45d9-48b8-b17e-6bc7acc37dc2'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"skill_name":"plan-consultation"}',
                    'call_id': 'call_1c695328d3f7406a8840085fa40b2dab',
                    'name': 'load_skill',
                    'type': 'function_call',
                    'id': 'fc_0cf3302c3ec7ef10016accf11d8f388191b04e3a2e1e31522f',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0cf3302c3ec7ef10016accf11ced8c8191a545374ef637f64b',
                'created_at': 1791815965.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0cf3302c3ec7ef10016accf11ced8c8191a545374ef637f64b',
            tool_calls=[
                {
                    'name': 'load_skill',
                    'args': {'skill_name': 'plan-consultation'},
                    'id': 'call_1c695328d3f7406a8840085fa40b2dab',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4685,
                'output_tokens': 22,
                'total_tokens': 4707,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='已加载技能 plan-consultation。正文中的相对路径可以用 read_skill_file 读取。\n\n# 
套餐咨询\n\n帮助用户在试用版、基础版、专业版、企业版中，选出满足需求且价格最低的套餐。\n\n## 执行步骤\n\n1. 
确认三项信息：团队成员人数、每月 AI 问答次数、是否需要 API 调用。\n   
缺少任何一项时，先向用户追问缺少的信息，不要读取资料，也不要推荐套餐。\n2. 三项信息齐全后，读取 
[套餐选型指南](references/plan-guide.md)，查出各套餐的成员人数上限、月度 AI 问答额度和 API 调用额度。\n3. 
从试用版开始，按价格从低到高逐个比较，选出第一个三项需求都满足的套餐。专业版也不满足时，推荐企业版，并说明需要联系
销售。\n4. 按“单价 × 成员人数”估算月费。企业版不公开标价，不做估算。\n\n## 输出要求\n\n- 推荐套餐：套餐名称\n- 
选择理由：逐项对比三项需求与套餐的上限或额度\n- 月费估算：写出计算式\n- 
需要人工确认：选型指南没有写明的内容\n\n只依据选型指南中的数字回答，不要凭印象补充价格或额度。',
            name='load_skill',
            id='75828e06-6ab8-4967-b254-6c6abf40587d',
            tool_call_id='call_1c695328d3f7406a8840085fa40b2dab'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"skill_name":"plan-consultation","path":"references/plan-guide.md"}',
                    'call_id': 'call_gwEnPC5NpJo4VSYPrQ8h7lkG',
                    'name': 'read_skill_file',
                    'type': 'function_call',
                    'id': 'fc_1791815968882_9pphakcegi',
                    'status': 'completed'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'gen_01M4R8VRV8B849C30C5M743DXX',
                'created_at': 1791815968.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='gen_01M4R8VRV8B849C30C5M743DXX',
            tool_calls=[
                {
                    'name': 'read_skill_file',
                    'args': {'skill_name': 'plan-consultation', 'path': 'references/plan-guide.md'},
                    'id': 'call_gwEnPC5NpJo4VSYPrQ8h7lkG',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 649,
            

[AIMessage] 输入 4685 token，请求 load_skill {"skill_name": "plan-consultation"}
[ToolMessage] load_skill 返回 485 字符
[AIMessage] 输入 649 token，请求 read_skill_file {"skill_name": "plan-consultation", "path": "references/plan-guide.md"}
[ToolMessage] read_skill_file 返回 513 字符
[AIMessage] 输入 5383 token，回答:
**推荐套餐：专业版。**

- **成员人数：**12 人，基础版上限 10 人；专业版上限 50 人。
- **AI 问答：**每月约 2 万次，基础版额度 5,000 次；专业版额度 30,000 次。
- **API 调用：**需要调用 API；基础版每月 10,000 次，专业版每月 80,000 次。

**月费估算：**199 元/人/月 × 12 人 = **2,388 元/月**。

API 的预估月调用量未提供，建议确认是否在专业版每月 80,000 次额度内。


这次三层都进入了上下文：先加载正文，再按第 2 步读取选型指南，最后逐项比较。基础版的人数上限（10 人）和问答额度（5000 次）都不够，专业版是第一个都满足的套餐，月费为 199 × 12 = 2388 元。

## 1.6 全量加载与按需加载的对比

如果不用加载器，最直接的做法是把所有技能的正文和资料都写进系统提示（**全量加载**）。下面按字符数比较两种方式在每个任务中进入上下文的内容：按需加载 = 系统提示 + 加载工具返回的内容；全量加载 = 包含全部正文和资料的系统提示。

In [37]:
def build_full_prompt(skills: dict[str, dict]) -> str:
    """把每个技能的正文和全部辅助文件直接拼进系统提示。"""
    parts = [BASE_PROMPT]
    for name, info in skills.items():
        _, body = split_skill_file(info["dir"] / "SKILL.md")
        parts.append(f"## 技能 {name}\n{body}")
        for extra in sorted(info["dir"].rglob("*")):
            if extra.is_file() and extra.name != "SKILL.md":
                relative = extra.relative_to(info["dir"])
                parts.append(f"## {name}/{relative}\n{extra.read_text(encoding='utf-8')}")
    return "\n\n".join(parts)


FULL_PROMPT = build_full_prompt(SKILLS)

print("全量加载的系统提示字符数:", len(FULL_PROMPT))
for label, stats in [("无关任务", unrelated_stats),
                     ("信息不全", partial_stats),
                     ("信息齐全", complete_stats)]:
    on_demand = len(SYSTEM_PROMPT) + stats["loaded_chars"]
    print(f"{label}：load_skill {stats['load_skill']} 次，read_skill_file "
          f"{stats['read_skill_file']} 次，按需加载 {on_demand} 字符，全量加载 {len(FULL_PROMPT)} 字符")

全量加载的系统提示字符数: 20344
无关任务：load_skill 0 次，read_skill_file 0 次，按需加载 437 字符，全量加载 20344 字符
信息不全：load_skill 1 次，read_skill_file 0 次，按需加载 922 字符，全量加载 20344 字符
信息齐全：load_skill 1 次，read_skill_file 1 次，按需加载 1435 字符，全量加载 20344 字符


三个任务的结果不同：

- 无关任务只多了一行技能摘要，比全量加载少得多。
- 信息齐全的任务需要全部内容，按需加载反而略多：系统提示中的技能摘要与使用规则、`load_skill` 返回的说明句都是额外内容，还多了两次工具调用的往返。
- 本节只有一个小技能，差距有限。技能越多，每个任务用不到的正文和资料越多，按需加载节省得越明显；而元数据层随技能数量增长，所以 `description` 要写得简短准确。

这里比较的是一次运行中进入上下文的内容。每次模型请求都会重新发送之前的消息，实际计费以 `usage_metadata` 的 token 数为准。

### 补充：官方的两种封装

本节的加载器与 LangChain 文档的 Skills 模式一致：系统提示列出技能，自定义 `load_skill` 工具按需返回内容。官方的 SQL 助手教程把同样的步骤封装成自定义中间件 `SkillMiddleware`：在 `wrap_model_call` 中把技能列表追加到系统消息，并通过中间件的 `tools` 属性注册 `load_skill`（中间件写法见第八章）。Deep Agents 则内置了 `SkillsMiddleware`，直接扫描 `SKILL.md` 目录，放在大纲的兴趣拓展中。

也就是将skill相关的提示词通过中间件注入

## 1.7 小结与小练习

| 层次 | 内容 | 何时进入上下文 | 本节实现 |
|---|---|---|---|
| 第 1 层 元数据 | `name`、`description` | 每次请求都在 | `discover_skills` 检查后写进系统提示 |
| 第 2 层 正文 | 执行步骤、输出要求 | 任务相关时 | `load_skill` 返回正文 |
| 第 3 层 资料 | `references/` 等文件 | 正文要求且确实需要时 | `read_skill_file` 在技能目录内读取 |

小练习：

1. 把 `description` 改成“套餐相关”，重新发现技能并创建 Agent，再运行任务 2。观察模型是否仍会加载技能，说明 `description` 写得具体有什么作用。
2. 删去 `SKILL.md` 第 1 步中“缺少任何一项时……”这句，重新运行任务 2。观察模型是否开始读取资料或直接推荐套餐。
3. 直接调用 `read_skill_file` 读取 `references/../SKILL.md`。预测是否允许，并用 `resolve()` 的结果解释。
4. 摘取知识库“三、额度与超额计费规则”，在 `references/` 中新增 `overage-rules.md`，并在 `SKILL.md` 中写明何时读取。设计一个会读取它的问题，以及一个不会读取它的问题，分别验证。

下一节综合实战会再编写一个退款审核技能，并把技能、第十一章的 MCP 服务与人工审核组合起来。

参考：[Agent Skills 规范](https://agentskills.io/specification)、[LangChain Skills 模式](https://docs.langchain.com/oss/python/langchain/multi-agent/skills)、[LangChain 教程：按需加载技能的 SQL 助手](https://docs.langchain.com/oss/python/langchain/multi-agent/skills-sql-assistant)、[Deep Agents Skills](https://docs.langchain.com/oss/python/deepagents/skills)。

In [38]:
# 练习3: 仅仅检查文件

skill_name = "plan-consultation"
path = "references/../SKILL.md"

skill_dir = SKILLS[skill_name]["dir"].resolve()
target = (skill_dir / path).resolve()

print("原始路径:", skill_dir / path)
print("resolve 后:", target)
print("仍在技能目录内:", target.is_relative_to(skill_dir))
print("目标是文件:", target.is_file())

# result = read_skill_file.invoke({
#     "skill_name": skill_name,
#     "path": path,
# })
# print(result[:200])

原始路径: /Users/skk/Developer/lang-chain-learn/Chapter12_skill/skills/plan-consultation/references/../SKILL.md
resolve 后: /Users/skk/Developer/lang-chain-learn/Chapter12_skill/skills/plan-consultation/SKILL.md
仍在技能目录内: True
目标是文件: True
